In [1]:
!pip install ultralytics pandas
!pip install -U ultralytics kagglehub pandas

import os
import pandas as pd
import kagglehub
from ultralytics import YOLO

print("📥 Veri seti indiriliyor...")
path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
print(f"✅ İndirme yolu: {path}")


data_path = path # Varsayılan
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        data_path = root
        break

print(f"📂 YOLO için ayarlanan veri yolu: {data_path}")
print(f"   İçerik: {os.listdir(data_path)}")

models_to_test = ['yolov8s-cls.pt', 'yolov8m-cls.pt']
learning_rates = [1e-3, 1e-4] # Hiperparametre testi

experiment_results = []

print("\n🚀 YOLOv8 Otomatik Hiperparametre Testi Başlıyor (Kaggle)...\n")

for model_name in models_to_test:
    for lr in learning_rates:
        
        run_name = f"{model_name.split('.')[0]}_lr{str(lr).replace('.', '')}"
        
        print(f"\n--- Eğitim Başlıyor: Model={model_name}, LR={lr} ---")
        
        model = YOLO(model_name) 
        
        # EĞİTİM
        results = model.train(
            data=data_path,
            epochs=10,
            imgsz=224,
            batch=16,
            lr0=lr, # Başlangıç Learning Rate
            project="/kaggle/working/PlantDoc_YOLO_Project",
            name=run_name,
            verbose=True
        )
        
        # SONUÇLARI ALMA
        try:
            # Eğitim sonrası validation metriklerini al
            metrics = model.val()
            # Top1 Accuracy değerini al ve float'a çevirip 4 basamak yuvarla
            top1_acc = round(float(metrics.top1), 4)
        except Exception as e:
            print(f"Metrik hatası: {e}")
            top1_acc = 0.0

        experiment_results.append({
            "Model": model_name,
            "Learning Rate": lr,
            "Top1 Accuracy": top1_acc,
            "Epochs": 10
        })

print("\n✅ Tüm Eğitimler Tamamlandı!")

df_results = pd.DataFrame(experiment_results)

df_results = df_results.sort_values(by="Top1 Accuracy", ascending=False)

print("\n" + "="*40)
print("TABLO 2: YOLOv8 Deney Sonuçları")
print("="*40)
print(df_results.to_markdown(index=False))

# CSV Olarak Kaydetme
save_path = "/kaggle/working/Tablo2_Sonuclar.csv"
df_results.to_csv(save_path, index=False)
print(f"\nSonuçlar '{save_path}' yoluna kaydedildi")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 124.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.0/160.0 kB 10.7 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: kagglehub
    Found existing installation: kagglehub 0.3.13
    Uninstalling kagglehub-0.3.13:
      Successfully uninstalled kagglehub-0.3.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
